# 01 - Tokens And Context Windows

## Learning objectives

By the end of this notebook, students should be able to:

- Explain why language models do not read text as characters or words directly.
- Estimate token counts and connect tokens to context windows and cost.
- Compare OpenAI and Anthropic message shapes without exposing API keys.
- Explain why chunking is necessary before retrieval-augmented generation.

## Concept primer

A token is a unit the model processes. It may be a word, part of a word, punctuation, whitespace pattern, or another learned text fragment. Context windows are measured in tokens, so tokenization affects cost, latency, and how much evidence can fit into a prompt.

In this course, the exact tokenizer is less important than the mental model: every prompt becomes a limited sequence of tokens, and every RAG system must choose what evidence deserves that limited space.


## Step 1 - Secure environment check

Run this cell first in every notebook. It loads `.env` files from either the repo root or this module folder. Notice that the code reports whether keys are loaded but never prints the keys themselves.

Students should create a local `.env` file with:

```bash
OPENAI_API_KEY=your_openai_api_key_here
ANTHROPIC_API_KEY=your_anthropic_api_key_here
LIVE_API=false
```

Set `LIVE_API=true` only when you intentionally want paid API calls.


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - Estimate token pressure

This simple approximation is deliberately imperfect. That is the teaching point: rough estimates are useful for planning, but production systems should use provider tokenizers or API usage metadata for billing and limits.


In [ ]:
from nlp_transformers_embeddings.utils.retrieval import approximate_token_count

examples = [
    "Transformers use attention to connect tokens across a sequence.",
    "RAG systems retrieve evidence before asking a model to answer.",
    "API keys should stay in local environment variables, not notebooks.",
]

for text in examples:
    # Approximate count helps students reason about prompt budget before live API calls.
    print(f"{approximate_token_count(text):>3} estimated tokens | {text}")


## What just happened?

We converted text length into a rough token budget. In production, this estimate informs chunk size, maximum answer length, and when to summarize or retrieve less context.


## Step 3 - Compare provider message shapes

OpenAI and Anthropic are both chat/generation providers, but their request shapes differ. Teaching both shapes helps students avoid assuming all LLM APIs are interchangeable.


In [ ]:
openai_messages = [
    {"role": "system", "content": "You are a concise NLP tutor."},
    {"role": "user", "content": "Explain why chunking matters in RAG."},
]

anthropic_request = {
    "system": "You are a concise NLP tutor.",
    "messages": [
        {"role": "user", "content": "Explain why chunking matters in RAG."}
    ],
}

print("OpenAI-style messages:")
print(openai_messages)
print("\nAnthropic Messages API style:")
print(anthropic_request)


## Step 4 - Chunking preview

Chunking splits long documents into retrieval-sized pieces. The overlap keeps nearby context together so an answer is less likely to miss facts that sit at a boundary.


In [ ]:
from nlp_transformers_embeddings.utils.mock_data import SAMPLE_DOCUMENTS
from nlp_transformers_embeddings.utils.retrieval import chunk_documents

chunks = chunk_documents(SAMPLE_DOCUMENTS, chunk_size_words=35, overlap_words=8)

for chunk in chunks[:4]:
    # Metadata travels with each chunk so retrieved evidence can be cited later.
    print(f"{chunk.chunk_id} | {chunk.title} | words {chunk.start_word}-{chunk.end_word}")
    print(chunk.text[:180] + "...\n")


## Student exercise

Change `chunk_size_words` and `overlap_words`. Observe how the number and shape of chunks changes. Then answer: what failure might happen if chunks are too small? What failure might happen if chunks are too large?

## Debugging checklist

- If imports fail, restart the kernel and run the setup cell first.
- If `.env` is missing, the notebook still runs in mock mode.
- If live calls are unexpectedly skipped, check that `LIVE_API=true` is set.

## Production best practices

- Never print API keys.
- Use actual tokenizer counts for provider-specific limits.
- Keep chunk metadata for citations.
- Treat context as scarce space, not an infinite text box.

## Reflection questions

1. Why is context-window management a product design problem, not just a technical limit?
2. Why does RAG usually chunk documents before embedding them?
3. What information should travel with every chunk?
